# OffScript — Phase 1: Data Quality Check

This notebook validates and cleans the combined Statcast dataset produced in
notebook 02 before it is used for modeling in Phase 2.

**Goals:**
- Audit missing values across all retained columns
- Identify and remove rare or unclassified pitch type codes
- Flag and remove physically implausible pitch locations
- Confirm all pitchers have sufficient volume for reliable modeling
- Persist the cleaned dataset for downstream use

**Input:** `data/processed/pitcher_data.parquet`  
**Output:** `data/processed/pitcher_data_clean.parquet`

## 1. Setup

In [1]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pitch_analysis import load_data

data = load_data()

print(f"Total pitches loaded: {len(data):,}")
print(f"Pitchers:             {data['pitcher_name'].nunique()}")
print(f"Date range:           {data['game_date'].min()} to {data['game_date'].max()}")

Total pitches loaded: 74,170
Pitchers:             14
Date range:           2023-03-30 to 2024-10-30


## 2. Missing Value Audit

Statcast records can be incomplete for a variety of reasons — tracking system
dropouts, pitch classification failures, or incomplete game logs. Columns with
high missingness rates may need to be dropped or imputed before modeling.

In [2]:
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(2)

missing_report = (
    pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
    .query("missing_count > 0")
    .sort_values("missing_pct", ascending=False)
)

print("=== Missing Value Report ===")
print(missing_report)

=== Missing Value Report ===
               missing_count  missing_pct
on_3b                  68517        92.38
on_2b                  62837        84.72
events                 54926        74.05
on_1b                  53807        72.55
plate_x                 1126         1.52
pitch_type              1126         1.52
pitch_name              1126         1.52
plate_z                 1126         1.52
release_speed           1118         1.51
pfx_x                   1118         1.51
pfx_z                   1118         1.51


## 3. Pitch Type Audit

Statcast occasionally logs unknown (`UN`) or transitional pitch type codes that
cannot be reliably modeled. Pitch types with fewer than 50 occurrences across
the full dataset are flagged for removal — too few examples to estimate
conditional probabilities with any confidence.

In [3]:
print("=== Pitch Type Distribution ===")
print(data["pitch_type"].value_counts())

RARE_PITCH_THRESHOLD = 50
pitch_counts = data["pitch_type"].value_counts()
rare_pitches = pitch_counts[pitch_counts < RARE_PITCH_THRESHOLD].index.tolist()

print(f"\nRare pitch types (< {RARE_PITCH_THRESHOLD} occurrences): {rare_pitches}")

=== Pitch Type Distribution ===
pitch_type
FF    23794
SL    10846
SI     8758
CH     8711
CU     7184
FC     7085
ST     3281
FS     1847
KC     1532
PO        6
Name: count, dtype: int64

Rare pitch types (< 50 occurrences): ['PO']


## 4. Location Outlier Audit

Plate location values outside physically plausible bounds indicate tracking
errors rather than real pitches. The thresholds below are conservative —
no legitimate pitch would be logged more than 4 feet off the horizontal center
or below ground level.

In [4]:
print("=== Location Ranges ===")
print(f"plate_x: {data['plate_x'].min():.2f} to {data['plate_x'].max():.2f}")
print(f"plate_z: {data['plate_z'].min():.2f} to {data['plate_z'].max():.2f}")

PLATE_X_LIMIT = 4.0
PLATE_Z_MIN = 0.0
PLATE_Z_MAX = 6.0

location_outliers = data[
    (data["plate_x"].abs() > PLATE_X_LIMIT)
    | (data["plate_z"] < PLATE_Z_MIN)
    | (data["plate_z"] > PLATE_Z_MAX)
]
print(f"\nLocation outliers flagged: {len(location_outliers)} pitches")

=== Location Ranges ===
plate_x: -4.24 to 3.86
plate_z: -3.23 to 11.02

Location outliers flagged: 941 pitches


## 5. Per-Pitcher Volume Check

Conditional probability estimates become unreliable below a minimum sample
size. Any pitcher with fewer than 500 pitches is flagged — this threshold
ensures enough observations per count state for the Phase 2 model to learn
meaningful patterns.

In [5]:
MIN_PITCH_VOLUME = 500

pitcher_counts = data.groupby("pitcher_name").size().sort_values()
print("=== Pitches Per Pitcher ===")
print(pitcher_counts)

low_volume = pitcher_counts[pitcher_counts < MIN_PITCH_VOLUME]
if len(low_volume) > 0:
    print(f"\nLow volume pitchers (< {MIN_PITCH_VOLUME} pitches): {low_volume.index.tolist()}")
else:
    print(f"\nAll pitchers exceed the {MIN_PITCH_VOLUME}-pitch modeling threshold")

=== Pitches Per Pitcher ===
pitcher_name
Max Scherzer        3283
Spencer Strider     3657
Nestor Cortes       4137
Justin Verlander    4461
Kyle Hendricks      4525
Chris Sale          4628
Gerrit Cole         5318
Marcus Stroman      5521
Yusei Kikuchi       5938
Framber Valdez      6014
Corbin Burnes       6366
Logan Webb          6590
Dylan Cease         6722
Zack Wheeler        7010
dtype: int64

All pitchers exceed the 500-pitch modeling threshold


## 6. Apply Cleaning and Save

Cleaning decisions based on the audits above:

| Issue | Decision |
|---|---|
| Missing `pitch_type` | Drop row — cannot model without a target label |
| Rare pitch type codes | Drop rows — too few samples for reliable estimates |
| Location outliers | Drop rows — tracking errors, not real pitches |
| Missing baserunner columns | Fill with 0 — absence of a runner ID means base is empty |

The cleaned dataset is persisted to parquet and consumed by all Phase 2
notebooks via `load_clean_data()` in `src/pitch_analysis.py`.

In [6]:
data_clean = data.copy()

# Drop rows with no pitch type — required target label for modeling
data_clean = data_clean.dropna(subset=["pitch_type"])

# Remove rare pitch types identified above
if rare_pitches:
    data_clean = data_clean[~data_clean["pitch_type"].isin(rare_pitches)]

# Remove physically implausible pitch locations
data_clean = data_clean[
    (data_clean["plate_x"].abs() <= PLATE_X_LIMIT)
    & (data_clean["plate_z"] >= PLATE_Z_MIN)
    & (data_clean["plate_z"] <= PLATE_Z_MAX)
]

# Fill missing baserunner columns — no runner ID means base is empty
for col in ("on_1b", "on_2b", "on_3b"):
    data_clean[col] = data_clean[col].fillna(0)

print(f"Pitches before cleaning: {len(data):,}")
print(f"Pitches after cleaning:  {len(data_clean):,}")
print(f"Pitches removed:         {len(data) - len(data_clean):,}")

OUTPUT_PATH = "../data/processed/pitcher_data_clean.parquet"
data_clean.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved: {OUTPUT_PATH}")

Pitches before cleaning: 74,170
Pitches after cleaning:  72,098
Pitches removed:         2,072

Saved: ../data/processed/pitcher_data_clean.parquet
